# 7. Experimento combinatorio de clasificación

## 7.1. Diseño

El experimento cruza cada modelo base con cada estrategia de balanceo y cada método de optimización. El resultado es un producto, no una suma:

| Etapa | Alternativas | Detalle |
|---|---|---|
| Modelos base | 7 | k-NN, Naive Bayes, regresión logística (L1/L2), árbol de decisión, Random Forest, XGBoost, SVM |
| Balanceo de clases | 4 | Sin balanceo, SMOTE, ADASYN, `class_weight="balanced"` |
| Optimización de hiperparámetros | 4 | Grid Search, Random Search, bayesiana (Optuna), genética |
| **Total** | **7 × 4 × 4 = 112** | Cada combinación se entrena, optimiza y evalúa de forma independiente |

Cada una de las 112 corridas atraviesa la validación anidada del capítulo 6: cinco folds externos para estimar desempeño, tres internos para seleccionar hiperparámetros, y agrupamiento por paciente en ambos niveles. Las métricas que se reportan provienen siempre del bucle externo.

### 7.1.1. Equivalentes de `class_weight`

Dos de los siete modelos no implementan el parámetro `class_weight`, y la guía exige registrar explícitamente lo que ocurre con esas combinaciones en lugar de omitirlas en silencio:

| Modelo | Tratamiento | Justificación |
|---|---|---|
| Naive Bayes | `priors=[0.5, 0.5]` | Fijar probabilidades a priori uniformes elimina la ventaja estructural de la clase mayoritaria en el producto posterior. Es el equivalente exacto del reponderado |
| XGBoost | `scale_pos_weight` | El parámetro multiplica el gradiente de la clase positiva; se fija en la razón entre clases del conjunto de entrenamiento |
| k-NN | **No aplicable** | Su predicción es un voto entre vecinos, sin término de pérdida que ponderar. No existe equivalente sin alterar el algoritmo, así que las cuatro combinaciones con `class_weight` se registran como no aplicables |

Eso deja 108 corridas ejecutables y 4 celdas documentadas como imposibles, distinción que el capítulo 10 necesita para aplicar la prueba de Friedman sobre un diseño completo.

In [ ]:
from config import *

import experimento as ex

train = leer_tabla("diabetes_train")
roles = roles_variables()

OBJETIVO = roles["objetivo"]
IDENTIFICADOR = roles["identificador"]
PREDICTORES = (roles["numericas"] + roles["ordinales"]
               + roles["binarias"] + roles["categoricas"])

X = train[PREDICTORES]
y = train[OBJETIVO]
grupos = train[IDENTIFICADOR]

# Razón entre clases, necesaria para el scale_pos_weight de XGBoost.
RAZON_DESBALANCE = float((1 - y.mean()) / y.mean())

print(f"Entrenamiento    : {len(X):,} encuentros de {grupos.nunique():,} pacientes")
print(f"Clase positiva   : {y.mean():.2%}")
print(f"Razón desbalance : {RAZON_DESBALANCE:.2f} a 1")
print(f"Modelos          : {ex.MODELOS}")
print(f"Balanceos        : {ex.BALANCEOS}")
print(f"Optimizadores    : {ex.OPTIMIZADORES}")

## 7.2. Presupuesto y multi-fidelidad

El capítulo 6 midió el costo por ajuste de cada familia y proyectó el experimento completo en unas 24 horas de cómputo, como cota inferior. Se aplican tres medidas, todas declaradas antes de ejecutar y registradas en la tabla maestra.

**Presupuesto por modelo.** Cada modelo evalúa tantas configuraciones como tiene su rejilla: 12 en los económicos y 8 en los costosos (k-NN, Random Forest y XGBoost). El presupuesto es **idéntico para los cuatro optimizadores de un mismo modelo**, de modo que la comparación entre optimizadores del capítulo 8 sea a igualdad de evaluaciones. Entre modelos, en cambio, introduce una asimetría: un modelo con 8 configuraciones explora menos su espacio que uno con 12, y eso lo penaliza. La columna `presupuesto` de la tabla maestra permite tenerlo en cuenta al comparar.

**Multi-fidelidad, primer mecanismo: Successive Halving.** Viene del motor y solo afecta a Optuna. La fidelidad es el número de folds internos evaluados: una configuración que tras el primer fold no está en el tercio superior se descarta sin evaluar los dos restantes (factor de reducción η = 3). La columna `n_ajustes` registra cuántos ajustes consumió realmente cada corrida, lo que permite medir el ahorro en el capítulo 8.

**Multi-fidelidad, segundo mecanismo: submuestreo de pacientes.** La búsqueda interna de los modelos costosos se ejecuta sobre el **25 % de los pacientes** del fold externo de entrenamiento (factor de reducción 4); el ajuste final de cada fold externo usa siempre todos sus datos, de modo que las métricas reportadas no pierden precisión. La submuestra se toma por paciente y no por fila, así que la integridad de los grupos se mantiene también en la búsqueda. Se aplica igual a los cuatro optimizadores del mismo modelo, por la misma razón que el presupuesto.

El factor 4 se elige para dejar unos 10,000 pacientes en la búsqueda, suficientes para estimar el AUC-PR con una precisión del orden de 0.01, que es la escala de las diferencias entre configuraciones. La fracción queda registrada en la columna `fraccion_busqueda`.

El supuesto que introduce el submuestreo es que el orden relativo entre configuraciones se conserva al reducir la muestra. No se da por bueno: el capítulo 8 lo verifica de forma directa, repitiendo la búsqueda de los modelos económicos con el 25 % y comparando la configuración elegida con la que sale del fold completo.

In [ ]:
# Modelos cuyo costo por ajuste justifica reducir la fidelidad de la búsqueda
# (medición en el capítulo 6, secciones 6.7 y 6.9).
COSTOSOS = {"random_forest", "xgboost", "knn"}

FIDELIDAD_ECONOMICA, FIDELIDAD_COSTOSA = 1.0, 0.25


def construir_corridas(modelos=None, balanceos=None, optimizadores=None):
    """Genera las combinaciones del diseño con su fidelidad de búsqueda.

    El presupuesto no se fija aquí: lo deriva el motor del tamaño de la
    rejilla de cada modelo (`ex.presupuesto_modelo`), de modo que rejilla y
    presupuesto no puedan desincronizarse.

    Parameters
    ----------
    modelos, balanceos, optimizadores : list of str, optional
        Subconjuntos a generar. Por omisión, el diseño completo.

    Returns
    -------
    list of tuple
        Pares (Corrida, fracción de búsqueda).
    """
    modelos = modelos or ex.MODELOS
    balanceos = balanceos or ex.BALANCEOS
    optimizadores = optimizadores or ex.OPTIMIZADORES

    corridas = []
    for modelo in modelos:
        costoso = modelo in COSTOSOS
        for balanceo in balanceos:
            for optimizador in optimizadores:
                corridas.append((
                    ex.Corrida(
                        modelo=modelo, balanceo=balanceo,
                        optimizador=optimizador,
                        razon_desbalance=RAZON_DESBALANCE,
                    ),
                    FIDELIDAD_COSTOSA if costoso else FIDELIDAD_ECONOMICA,
                ))
    return corridas


diseno = construir_corridas()
resumen_diseno = pd.DataFrame([
    {"modelo": c.modelo, "balanceo": c.balanceo,
     "optimizador": c.optimizador,
     "presupuesto": ex.presupuesto_modelo(c.modelo),
     "fidelidad": f}
    for c, f in diseno
])
print(f"Corridas del diseño: {len(resumen_diseno)}")
resumen_diseno.groupby("modelo").agg(
    corridas=("modelo", "size"),
    presupuesto=("presupuesto", "first"),
    fidelidad=("fidelidad", "first"),
)

## 7.3. Ejecución

El ejecutor escribe cada corrida en la tabla maestra en cuanto termina y omite las ya presentes al reanudar, de modo que el experimento pueda repartirse en varias sesiones sin perder trabajo. La celda siguiente se puede interrumpir y volver a lanzar sin consecuencias.

Se ejecuta por bloques, de los modelos económicos a los costosos, para tener resultados parciales interpretables desde el principio. Cambia `BLOQUES_A_EJECUTAR` para lanzar solo una parte.

In [ ]:
RUTA_TABLA = RESULTADOS / "tabla_maestra_clasificacion.csv"

BLOQUES = {
    "economicos": ["bayes", "logistica", "arbol", "svm"],
    "costosos": ["knn", "random_forest", "xgboost"],
}

# Ajusta esta lista para ejecutar por partes. Con [] no se ejecuta nada y solo
# se analiza lo que ya esté en la tabla maestra.
BLOQUES_A_EJECUTAR = ["economicos", "costosos"]

In [ ]:
import time

for nombre_bloque in BLOQUES_A_EJECUTAR:
    modelos = BLOQUES[nombre_bloque]
    print(f"\n{'=' * 70}\nBloque: {nombre_bloque} ({', '.join(modelos)})\n"
          f"{'=' * 70}")
    inicio = time.perf_counter()

    for corrida, fidelidad in construir_corridas(modelos=modelos):
        ex.ejecutar_experimento(
            [corrida], X, y, grupos, roles,
            ruta_tabla=RUTA_TABLA,
            folds_externos=5, folds_internos=3,
            metrica="average_precision",
            fraccion_busqueda=fidelidad,
        )

    print(f"\nBloque {nombre_bloque} terminado en "
          f"{(time.perf_counter() - inicio) / 60:.1f} min")

La métrica que guía la búsqueda interna es la **precisión media** (AUC-PR). La razón es el desbalance: con un 11.4 % de clase positiva, el AUC-ROC se calcula sobre un eje de falsos positivos normalizado por una clase mayoritaria muy grande, lo que lo hace poco sensible a mejoras en la clase de interés. El AUC-PR, en cambio, responde directamente a la capacidad de identificar readmisiones sin inundar de falsos positivos la lista de seguimiento.

## 7.4. Resultados

In [ ]:
maestra = pd.read_csv(RUTA_TABLA)
completadas = maestra.query("estado == 'completada'").copy()

print(f"Corridas registradas : {len(maestra)}")
print(f"  completadas        : {len(completadas)}")
print(f"  no aplicables      : {(maestra['estado'] == 'no aplicable').sum()}")
print(f"Tiempo total acumulado: "
      f"{maestra['tiempo_total_s'].sum() / 3600:.2f} h")

# reindex, no indexación directa: mientras no haya ninguna fila no aplicable
# la columna "detalle" todavía no existe en la tabla maestra.
maestra.query("estado == 'no aplicable'").reindex(
    columns=["modelo", "balanceo", "optimizador", "detalle"])

In [ ]:
COLUMNAS_RESUMEN = [
    "modelo", "balanceo", "optimizador", "presupuesto", "fraccion_busqueda",
    "auc_pr_media", "auc_pr_sd", "auc_roc_media", "auc_roc_sd",
    "recall_media", "precision_media", "f1_media",
    "exactitud_balanceada_media", "brier_media",
    "n_evaluaciones", "n_ajustes", "tiempo_por_evaluacion_s",
    "tiempo_busqueda_s", "tiempo_ajuste_s", "tiempo_inferencia_s",
    "semilla",
]

ranking = completadas.sort_values("auc_pr_media", ascending=False)
tabla(ranking[COLUMNAS_RESUMEN].round(4), filas=25)

In [ ]:
# Mejor configuración de cada modelo, con su desviación estándar entre folds.
mejores = (completadas.loc[completadas.groupby("modelo")["auc_pr_media"]
                                     .idxmax()]
                      .sort_values("auc_pr_media", ascending=False))

fig, ax = plt.subplots(figsize=(7.5, 3.6))
posiciones = np.arange(len(mejores))
ax.barh(posiciones, mejores["auc_pr_media"], xerr=mejores["auc_pr_sd"],
        color=PALETA[1], error_kw={"ecolor": "black", "capsize": 3,
                                   "elinewidth": 0.9})
ax.set_yticks(posiciones)
ax.set_yticklabels(mejores["modelo"])
ax.invert_yaxis()
ax.axvline(y.mean(), ls="--", color=GRIS, lw=1,
           label=f"Prevalencia ({y.mean():.3f})")
ax.set_xlabel("AUC-PR (media ± desviación estándar entre folds externos)")
ax.set_title("Mejor configuración por modelo")
ax.legend()
plt.tight_layout()
plt.show()

guardar_resultado(mejores.set_index("modelo")[COLUMNAS_RESUMEN[1:]],
                  "mejores_por_modelo")
mejores[["modelo", "balanceo", "optimizador", "auc_pr_media", "auc_pr_sd",
         "auc_roc_media", "tiempo_ajuste_s"]].round(4)

La línea discontinua marca la prevalencia de la clase positiva, que es el AUC-PR que obtendría un clasificador aleatorio. La distancia de cada barra a esa línea es la señal real que captura el modelo.

Al leer esta figura conviene comparar las diferencias entre modelos con las barras de error. Si los intervalos se solapan ampliamente, la diferencia observada puede ser ruido de partición y no una ventaja real: es precisamente la pregunta que el capítulo 10 responde con pruebas formales en lugar de con inspección visual.

Las columnas de costo (`n_evaluaciones`, `n_ajustes` y `tiempo_por_evaluacion_s`) se incluyen en el ranking porque son las que permiten leer cada resultado junto a lo que costó obtenerlo. Su comparación sistemática entre optimizadores corresponde al capítulo 8.

### 7.4.1. Efecto de las estrategias de balanceo

In [ ]:
comparacion_balanceo = (completadas.pivot_table(
    index="modelo", columns="balanceo", values="auc_pr_media",
    aggfunc="max"))
orden_balanceo = [b for b in ex.BALANCEOS if b in comparacion_balanceo.columns]
comparacion_balanceo = comparacion_balanceo[orden_balanceo]

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(comparacion_balanceo, annot=True, fmt=".4f", cmap="RdYlGn",
            linewidths=0.5, cbar_kws={"label": "AUC-PR"}, ax=ax)
ax.set_title("AUC-PR de la mejor configuración por modelo y balanceo")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

comparacion_balanceo.round(4)

In [ ]:
# Ganancia de cada estrategia frente a la ausencia de balanceo, por modelo.
if "ninguno" in comparacion_balanceo.columns:
    ganancia = comparacion_balanceo.sub(comparacion_balanceo["ninguno"],
                                        axis=0).drop(columns="ninguno")
    resumen_ganancia = pd.DataFrame({
        "ganancia media en AUC-PR": ganancia.mean(),
        "ganancia mínima": ganancia.min(),
        "ganancia máxima": ganancia.max(),
        "modelos donde mejora": (ganancia > 0).sum(),
    }).round(4)
    display(resumen_ganancia)
    display(ganancia.round(4))

In [ ]:
# El balanceo actúa sobre el umbral implícito, así que su efecto es mucho
# mayor en métricas dependientes del umbral que en las de ordenamiento.
efecto_umbral = completadas.pivot_table(
    index="balanceo", values=["auc_pr_media", "auc_roc_media", "recall_media",
                              "precision_media", "f1_media"],
    aggfunc="mean").reindex(orden_balanceo).round(4)
efecto_umbral

Esta última tabla es clave para interpretar el papel del balanceo y conviene discutirla con cuidado en el informe.

Las métricas de ordenamiento (AUC-PR y AUC-ROC) miden si el modelo asigna mayor riesgo a los pacientes que efectivamente reingresan, con independencia de dónde se ponga el umbral de decisión. Las métricas de clasificación (recall, precisión, F1) se calculan con umbral 0.5 y por tanto dependen de cómo el modelo distribuya sus probabilidades.

El balanceo desplaza esa distribución: sube las probabilidades predichas de la clase minoritaria y, con umbral fijo, aumenta el recall a costa de la precisión. Eso puede parecer una mejora si solo se mira el recall, pero en las métricas de ordenamiento el efecto suele ser mucho menor. La conclusión metodológica es que **el balanceo y la elección del umbral resuelven el mismo problema por vías distintas**, y que comparar modelos por recall con umbral 0.5 confunde ambos efectos. El capítulo 9 separa las dos decisiones eligiendo el umbral de forma explícita.

Un caso extremo que aparece en los resultados: sin balanceo y con umbral 0.5, varios modelos tienen recall cercano a cero, porque prácticamente ninguna probabilidad predicha supera 0.5 cuando la prevalencia es del 11 %. Su AUC-PR, en cambio, es competitivo. Eso demuestra que el umbral por defecto es inadecuado en este problema, no que los modelos sin balanceo sean malos.

### 7.4.2. Desempeño frente a costo

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for modelo, grupo in completadas.groupby("modelo"):
    ax.scatter(grupo["tiempo_ajuste_s"], grupo["auc_pr_media"],
               label=modelo, s=45, alpha=0.8)
ax.set_xscale("log")
ax.set_xlabel("Tiempo de ajuste acumulado en los 5 folds (s, escala log)")
ax.set_ylabel("AUC-PR medio")
ax.set_title("Desempeño frente a costo computacional")
ax.axhline(y.mean(), ls="--", color=GRIS, lw=1)
ax.legend(title="Modelo", fontsize=8, ncols=2)
plt.tight_layout()
plt.show()

Este gráfico responde a una pregunta que la guía plantea de forma explícita: si la complejidad añadida se traduce en mejora. Un modelo situado arriba y a la izquierda domina, porque logra más desempeño con menos cómputo. Si los ensambles aparecen a la derecha sin estar claramente más arriba que los modelos lineales, el coste no está justificado para este problema.

### 7.4.3. Estabilidad frente a la semilla

La media y la desviación estándar entre folds capturan la variabilidad de una partición concreta. Repetir la mejor configuración con otras semillas mide algo más amplio: la semilla gobierna a la vez el reparto en folds, la submuestra de la búsqueda y la aleatoriedad interna de los algoritmos (arranque de Optuna, población inicial del genético, remuestreo de SMOTE, árboles de Random Forest). El rango entre semillas es, por tanto, la variabilidad total del procedimiento de principio a fin, no solo la del algoritmo.

Esa es justamente la cifra que hace falta para interpretar el ranking: si dos combinaciones difieren menos de lo que varía una sola combinación al cambiar de semilla, la diferencia no es atribuible al método.

In [ ]:
SEMILLAS = [42, 7, 2026]
mejor_global = ranking.iloc[0]

filas = []
for semilla in SEMILLAS:
    corrida = ex.Corrida(
        modelo=mejor_global["modelo"], balanceo=mejor_global["balanceo"],
        optimizador=mejor_global["optimizador"],
        razon_desbalance=RAZON_DESBALANCE,
    )
    resultado = ex.ejecutar_corrida(
        corrida, X, y, grupos, roles, folds_externos=5, folds_internos=3,
        fraccion_busqueda=(FIDELIDAD_COSTOSA
                           if mejor_global["modelo"] in COSTOSOS
                           else FIDELIDAD_ECONOMICA),
        semilla=semilla,
    )
    filas.append({"semilla": semilla,
                  "auc_pr_media": resultado["auc_pr_media"],
                  "auc_pr_sd": resultado["auc_pr_sd"],
                  "auc_roc_media": resultado["auc_roc_media"]})

sensibilidad = pd.DataFrame(filas).set_index("semilla").round(4)
sensibilidad.loc["rango"] = (sensibilidad.max() - sensibilidad.min())
guardar_resultado(sensibilidad, "sensibilidad_semillas")
sensibilidad

La fila `rango` es la que importa: si la variación entre semillas es del mismo orden que las diferencias entre combinaciones del ranking, esas diferencias no son interpretables sin pruebas estadísticas. Es el argumento que motiva el capítulo 10.

## 7.5. Síntesis del capítulo

Al cerrar este capítulo conviene dejar por escrito, con los números de la tabla maestra:

| Pregunta | Dónde se responde |
|---|---|
| ¿Qué combinación obtiene el mejor AUC-PR y con qué desviación entre folds? | Ranking de la sección 7.4 |
| ¿Cuál es el mejor modelo de cada familia y qué distancia hay entre ellos? | Figura de mejores por modelo |
| ¿Aporta el balanceo en las métricas de ordenamiento, o solo desplaza el umbral? | Tablas de la sección 7.4.1 |
| ¿Justifican los ensambles su costo computacional frente a los modelos lineales? | Gráfico de desempeño frente a costo |
| ¿Son las diferencias entre combinaciones mayores que la variabilidad por semilla? | Sección 7.4.3, y de forma formal en el capítulo 10 |

Dos decisiones de diseño quedan declaradas y registradas en la tabla maestra, para que cualquier lectura del ranking las tenga presentes: el presupuesto es menor en los modelos costosos (8 configuraciones frente a 12) y su búsqueda usa el 25 % de los pacientes. Ambas afectan por igual a los cuatro optimizadores de un mismo modelo, de modo que no distorsionan la comparación del capítulo 8, pero sí penalizan a los modelos costosos frente a los económicos.

El capítulo 8 compara los cuatro métodos de optimización entre sí: calidad alcanzada por presupuesto, curvas de convergencia, diversidad genética, costo por evaluación y verificación del supuesto de la multi-fidelidad. El capítulo 9 evalúa en detalle la mejor combinación sobre el conjunto de prueba, con calibración, elección de umbral e interpretabilidad. El capítulo 10 aplica la secuencia estadística jerárquica sobre las métricas por fold.